# Notebook 2 of 3 — LangGraph

**Every line of code explained. Theory included. Runs on your OpenAI API key.**

| Notebook | Covers |
|---|---|
| 1. LangChain | models, messages, prompts, chains, structured output, RAG |
| **2. LangGraph** ← you are here | state, nodes, edges, memory, agents, human approval |
| 3. LangSmith | tracing, datasets, evaluation |

This notebook is **standalone** — §0 rebuilds the small amount of LangChain it needs. If you have done notebook 1, §0 will look familiar; skim it and move on.

---

## What LangGraph is, in one paragraph

LangGraph is an **orchestration framework**: you describe your application as a **graph** — nodes are steps, edges say what happens next — with a **shared state** flowing through it. Because the flow is an explicit graph rather than a hidden loop, it can **cycle** (retry, self-correct), **branch** (route to specialists), **persist** (survive a crash or a three-day gap), **stream** (show progress), and **pause** for a human to approve something.

### The problem it solves

In notebook 1 you built the agent loop by hand:

```python
while True:
    ai = llm_with_tools.invoke(history)
    if not ai.tool_calls: return ai.content
    for call in ai.tool_calls:
        history.append(ToolMessage(run(call), tool_call_id=call["id"]))
```

That works, and it is missing everything that production needs:

| Missing | Consequence |
|---|---|
| Persistence | the process dies at step 3 and all work is lost |
| Memory across requests | a follow-up tomorrow starts from zero |
| Human approval | it can issue a $349 refund with nobody watching |
| Branching | you cannot route billing tickets differently from technical ones |
| Streaming of steps | the user stares at a blank screen for 20 seconds |
| Resumability | you cannot rewind to step 2 and try another path |

📌 **The interview answer to "why not just a while loop?"**
> *"Because production agents need durable state, controlled branching, streaming and human-in-the-loop. LangGraph gives me a checkpointed state machine, so the loop is inspectable, resumable and pausable instead of a black box."*

### Chain vs graph

```
LangChain chain — linear, stateless          LangGraph — stateful, branching, cyclic

   prompt → model → parser                       START → classify → retrieve → draft
                                                                        ▲        │
                                                                        └─ review ┘
                                                                          │
                                                                    human approval ⏸
```

| Use a chain when… | Use a graph when… |
|---|---|
| steps are fixed and linear | you need loops, retries or self-correction |
| one LLM call, or simple RAG | multiple specialists must collaborate |
| stateless request → response | state must survive across requests or crashes |
| no human in the flow | a human must approve an action |

### Contents

| § | Topic |
|---|---|
| 0 | Setup + the LangChain pieces we need |
| 1 | The four primitives · your first graph (no LLM) |
| 2 | Nodes and partial state updates |
| 3 | Reducers and `add_messages` |
| 4 | Checkpointers and `thread_id` — real memory |
| 5 | Conditional edges — routing |
| 6 | The ReAct agent as a graph · `ToolNode` · `create_agent` |
| 7 | Human in the loop — `interrupt()` and `Command` |
| 8 | Streaming · visualising the graph |
| 9 | **Use case A** — support triage agent with a critic loop and a refund gate |
| 10 | **Use case B** — research assistant with a plan → gather → check → write cycle |
| 11 | Self-check, troubleshooting, recap |

## 0. Setup

No version pins — pinning two packages of the same family to guessed versions makes pip fail resolution, and pip is all-or-nothing, so *nothing* installs.

In [ ]:
%pip install -q -U langchain langchain-openai langchain-community langchain-text-splitters langgraph faiss-cpu

print("Install finished. If a 'RESTART SESSION' button appeared, click it and continue.")

In [ ]:
import os, getpass

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Paste your OpenAI API key (sk-...): ")

CHAT_MODEL = "gpt-4o-mini"
EMBED_MODEL = "text-embedding-3-small"

# Preflight: catch install problems now, not 20 cells later.
problems = []
try:
    from langchain_openai import ChatOpenAI, OpenAIEmbeddings
    from langgraph.graph import StateGraph
    from langgraph.checkpoint.memory import InMemorySaver
except ImportError as e:
    problems.append(str(e))

if problems:
    print("PROBLEMS:", problems, "\n-> re-run the install cell, restart the session, re-run this.")
else:
    llm = ChatOpenAI(model=CHAT_MODEL, temperature=0, timeout=30, max_retries=2)
    print("live test ->", llm.invoke("Reply with the single word: ready").content)
    print("ALL GOOD.")

In [ ]:
# The LangChain pieces this notebook needs. All of this is explained in notebook 1;
# it is repeated here so notebook 2 stands alone.
import textwrap
from langchain_openai import OpenAIEmbeddings
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.tools import tool
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pydantic import BaseModel, Field
from typing import Literal, Optional, List, TypedDict, Annotated

emb = OpenAIEmbeddings(model=EMBED_MODEL)

# --- a tiny policy knowledge base, so our agents have something real to look up ---
os.makedirs("kb", exist_ok=True)
kb_files = {
    "returns_policy.txt": """
    ACME RETURNS POLICY
    Standard consumers: return within 30 days for a full refund if unopened.
    Opened items are subject to a 15% restocking fee.
    Enterprise customers (above $50,000/year): 90-day window, restocking fee waived.
    Refunds reach the original payment method 5-7 business days after the return is scanned.
    """,
    "shipping_policy.txt": """
    ACME SHIPPING POLICY
    Standard shipping is free above $75 and takes 3-5 business days.
    Express shipping costs $19.99 and takes 1-2 business days.
    If tracking has not updated for 7 consecutive days the parcel is DELAYED.
    Support agents may reship the order free of charge, or issue a full refund,
    whichever the customer prefers.
    """,
    "warranty_policy.txt": """
    ACME WARRANTY TERMS
    All hardware carries a 24-month limited warranty covering manufacturing defects.
    Accidental damage such as drops and liquid spills is NOT covered.
    AcmeCare+ costs $89/year and adds accidental damage protection with a $49 deductible.
    """,
    "support_sla.txt": """
    ACME SUPPORT SLA
    Free tier: email only, first response within 72 hours.
    Pro tier: email and chat, first response within 24 hours.
    Enterprise tier: dedicated Slack channel, 1-hour first response in business hours,
    and 4 hours for P1 incidents at any time. A P1 is a full production outage.
    """,
}
for name, body in kb_files.items():
    with open(f"kb/{name}", "w", encoding="utf-8") as f:
        f.write(textwrap.dedent(body).strip())

docs = DirectoryLoader("kb", glob="**/*.txt", loader_cls=TextLoader,
                       loader_kwargs={"encoding": "utf-8"}).load()
chunks = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=100).split_documents(docs)
for c in chunks:
    c.metadata["file"] = os.path.basename(c.metadata["source"])

store = FAISS.from_documents(chunks, emb)
retriever = store.as_retriever(search_kwargs={"k": 3})

def format_docs(ds) -> str:
    """list[Document] -> one labelled string, so the model can cite sources."""
    return "\n\n---\n\n".join(f"[{d.metadata['file']}]\n{d.page_content}" for d in ds)

# --- a fake orders database and two tools ---
ORDERS = {
    "B-99182": {"item": "27-inch monitor",     "status": "DELAYED",    "days": 12, "value": 349.00},
    "A-10023": {"item": "laptop stand",        "status": "DELIVERED",  "days": 4,  "value": 45.50},
    "C-55510": {"item": "mechanical keyboard", "status": "IN_TRANSIT", "days": 2,  "value": 129.99},
}

@tool
def get_order_status(order_id: str) -> str:
    """Look up the shipping status of an order by its id, e.g. 'B-99182'."""
    o = ORDERS.get(order_id.strip().upper())
    if not o:
        return f"No order found with id {order_id}."
    return (f"Order {order_id}: {o['item']}, status {o['status']}, "
            f"shipped {o['days']} days ago, value ${o['value']:.2f}.")

@tool
def search_policies(query: str) -> str:
    """Search Acme's policy documents about returns, shipping, warranty and SLA."""
    return format_docs(retriever.invoke(query))

TOOLS = [get_order_status, search_policies]
llm_with_tools = llm.bind_tools(TOOLS)

print(f"setup done: {len(chunks)} chunks indexed, {len(TOOLS)} tools ready")

## 1. The four primitives, and your first graph

### Theory

| Primitive | What it is | Code |
|---|---|---|
| **State** | a shared dict, typed with `TypedDict`, that flows through the graph | `class S(TypedDict): value: int` |
| **Node** | a plain Python function: state in, **partial** update out | `def n(state): return {"value": 1}` |
| **Edge** | "after A, always go to B" | `builder.add_edge("a", "b")` |
| **Conditional edge** | a function that reads state and *chooses* the next node | `builder.add_conditional_edges("a", router, ["b","c"])` |

Plus two runtime superpowers, covered in §4 and §7:

- **Checkpointer** — saves state after every node, keyed by `thread_id`. Gives memory, crash recovery and time travel.
- **Interrupt** — pauses mid-run so a human can approve or edit, then resumes exactly where it stopped.

**That is all of LangGraph.** Everything else is a combination of those six things.

### The build sequence is always identical

```python
builder = StateGraph(State)          # 1. declare the state shape
builder.add_node("name", fn)         # 2. register nodes by name
builder.add_edge(START, "name")      # 3. wire them up
graph = builder.compile()            # 4. validate and get a Runnable
graph.invoke(initial_state, config)  # 5. run
```

We start with **no LLM at all**, so the mechanics are unmistakable.

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

# 1️⃣ STATE — the shared "whiteboard" every node can read and write.
#    TypedDict is a plain dict at runtime, with type hints for you, your IDE and LangGraph.
class CounterState(TypedDict):
    value: int      # a number we will modify
    log: str        # a text trail, so we can see what happened

# 2️⃣ NODES — ordinary functions. No base class, no decorator. Take state, return a dict.
def add_ten(state: CounterState) -> dict:
    new = state["value"] + 10
    # Returning {"value": new} MERGES into the state. Keys you omit stay untouched.
    return {"value": new, "log": state["log"] + f" +10={new}"}

def double(state: CounterState) -> dict:
    new = state["value"] * 2
    return {"value": new, "log": state["log"] + f" x2={new}"}

# 3️⃣ BUILD
builder = StateGraph(CounterState)      # needs the state class to validate node outputs
builder.add_node("add_ten", add_ten)    # register under a NAME - edges refer to these strings
builder.add_node("double", double)
builder.add_edge(START, "add_ten")      # START = the virtual entry point. Every graph needs one.
builder.add_edge("add_ten", "double")   # after add_ten, ALWAYS go to double
builder.add_edge("double", END)         # END = the virtual exit. Reaching it stops the run.

# 4️⃣ COMPILE — validates the topology and returns a Runnable
graph = builder.compile()

# 5️⃣ RUN — the same .invoke() as every LangChain component
result = graph.invoke({"value": 5, "log": "start"})
print(result)
print("\n(5 + 10) * 2 = 30 ✅")

# ⭐ .stream() shows every intermediate state. THIS is your debugger, and it is the
#    reason a graph beats a black-box agent: you can see what each step did.
print("\nstep by step:")
for step in graph.stream({"value": 1, "log": "start"}):
    print("  ", step)      # {node_name: the update that node returned}

## 2. Nodes and partial updates

A node is *any* callable with the signature `state → dict`. That is the whole contract.

| Node returns | Effect |
|---|---|
| `{"value": 30}` | sets `state["value"] = 30`, leaves everything else alone |
| `{}` | changes nothing (perfectly valid — useful for logging nodes) |
| the whole state | works, but noisy; prefer partial updates |
| a key not in the state class | error — this is why `TypedDict` exists |

Nodes can do anything: call an LLM, hit a database, run a regex, call another graph. There is nothing special about them, which is exactly why LangGraph is easy to test — **every node is a unit-testable function.**

In [ ]:
# Nodes are just functions, so you can test one directly - no graph, no LLM, no mocking.
print("add_ten alone:", add_ten({"value": 0, "log": ""}))
print("double alone :", double({"value": 3, "log": ""}))

# A node that returns {} changes nothing. Useful for a pure side effect like logging.
def audit(state: CounterState) -> dict:
    print(f"   [audit] value is currently {state['value']}")
    return {}          # no state change

b2 = StateGraph(CounterState)
b2.add_node("add_ten", add_ten)
b2.add_node("audit", audit)
b2.add_node("double", double)
b2.add_edge(START, "add_ten")
b2.add_edge("add_ten", "audit")     # slot the audit node into the middle
b2.add_edge("audit", "double")
b2.add_edge("double", END)

print("\nwith an audit node in the middle:")
print(b2.compile().invoke({"value": 5, "log": "start"})["value"])

## 3. Reducers and `add_messages`

### The problem

By default, a node returning `{"value": 30}` **overwrites** `state["value"]`. That is right for a counter.

But for a **chat history** you don't want to overwrite — you want to **append**. The rule that decides how an update merges with the old value is called a **reducer**.

You attach one with `Annotated`:

```python
messages: Annotated[list, add_messages]
```

Read it as: *"this field is a list, and combine updates using `add_messages`."*

### Why `add_messages` and not `list.append`

`add_messages` does three things:

1. **appends** new messages,
2. **converts** raw dicts like `{"role": "user", "content": "hi"}` into proper message objects,
3. **replaces** an existing message when the new one has the same `id` — which is how message editing and time travel work.

📌 **This is a top-3 LangGraph interview question.** Answer: *"State updates are merged by reducers. The default reducer overwrites; `add_messages` appends and de-duplicates by message id."*

You can write your own reducer too — §9 uses one for an audit log.

In [ ]:
from typing import Annotated
from langgraph.graph.message import add_messages

# ---------- DEFAULT reducer: OVERWRITE ----------
class OverwriteState(TypedDict):
    items: list                       # no Annotated -> the default reducer applies

def add_a(state): return {"items": ["a"]}
def add_b(state): return {"items": ["b"]}

ow = StateGraph(OverwriteState)
ow.add_node("a", add_a); ow.add_node("b", add_b)
ow.add_edge(START, "a"); ow.add_edge("a", "b"); ow.add_edge("b", END)
print("default reducer  :", ow.compile().invoke({"items": []}))     # -> ['b']  (a was WIPED)

# ---------- CUSTOM reducer: APPEND ----------
class AppendState(TypedDict):
    items: Annotated[list, lambda old, new: old + new]
    #                      ^ any function (old_value, new_value) -> merged_value

ap = StateGraph(AppendState)
ap.add_node("a", add_a); ap.add_node("b", add_b)
ap.add_edge(START, "a"); ap.add_edge("a", "b"); ap.add_edge("b", END)
print("custom reducer   :", ap.compile().invoke({"items": []}))     # -> ['a', 'b']

# ---------- add_messages: the built-in reducer for conversations ----------
class ChatState(TypedDict):
    messages: Annotated[list, add_messages]

# It also coerces plain dicts into message objects:
merged = add_messages(
    [HumanMessage("hello")],                                 # existing
    [{"role": "assistant", "content": "hi there"}],          # incoming, as a raw dict
)
print("\nadd_messages coerces dicts:")
for m in merged:
    print(f"  {m.type:<10} {type(m).__name__:<14} {m.content}")

## 4. Checkpointers and `thread_id` — real memory

### Theory

A **checkpointer** saves the state after **every node**. Combined with a `thread_id`, that gives you four things at once:

| Capability | How it follows from checkpointing |
|---|---|
| **Memory** | the next request loads the saved state instead of starting empty |
| **Crash recovery** | the process dies mid-run; the last checkpoint is still in the database |
| **Human-in-the-loop** | a "pause" is just a checkpoint you haven't resumed yet (§7) |
| **Time travel** | you can inspect or rewind to any earlier checkpoint |

A **thread** is one conversation: one user, one chat, one ticket. `thread_id` is how you say "which conversation is this?".

```
┌─────────────────────────────────────────────────────────────┐
│ Request 1  thread_id="alice"  →  graph runs  →  checkpoint  │
│                                                    │        │
│                                                    ▼        │
│                                            [ Postgres ]     │
│                                                    │        │
│ Request 2  thread_id="alice"  ←  history reloaded ─┘        │
│ (hours later, possibly a different server)                  │
└─────────────────────────────────────────────────────────────┘
```

| Checkpointer | Use for |
|---|---|
| `InMemorySaver` | notebooks, tests. It is literally a dict; dies with the process |
| `SqliteSaver` | one machine, real persistence (`pip install langgraph-checkpoint-sqlite`) |
| `PostgresSaver` | production (`pip install langgraph-checkpoint-postgres`) |

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

def chatbot(state: ChatState) -> dict:
    """One LLM turn. Reads the WHOLE history, returns ONE new message."""
    reply = llm.invoke(state["messages"])       # the state IS the message list
    return {"messages": [reply]}                # add_messages appends it - no bookkeeping

cb = StateGraph(ChatState)
cb.add_node("chatbot", chatbot)
cb.add_edge(START, "chatbot")
cb.add_edge("chatbot", END)

memory = InMemorySaver()                        # ⭐ the checkpointer
chat_graph = cb.compile(checkpointer=memory)

# The config is the SECOND argument to invoke. thread_id identifies the conversation.
alice = {"configurable": {"thread_id": "alice"}}
bob   = {"configurable": {"thread_id": "bob"}}

r1 = chat_graph.invoke({"messages": [HumanMessage("Hi, I'm Alice, a data analyst.")]}, alice)
print("alice:", r1["messages"][-1].content)

# ⭐ The second call sends ONLY the new message. The checkpointer reloads the history.
r2 = chat_graph.invoke({"messages": [HumanMessage("What's my name and my job?")]}, alice)
print("alice:", r2["messages"][-1].content)

# A different thread_id is a completely separate conversation - this is user isolation.
r3 = chat_graph.invoke({"messages": [HumanMessage("What's my name?")]}, bob)
print("bob  :", r3["messages"][-1].content)

# Inspect what was saved. get_state is the single most useful debugging call in LangGraph.
snap = chat_graph.get_state(alice)
print(f"\nalice thread holds {len(snap.values['messages'])} messages")
print(f"next node to run: {snap.next}   (empty = the run finished)")

### Line by line

| Line | What it does | Why |
|---|---|---|
| `llm.invoke(state["messages"])` | passes the whole history to the model | the state *is* the conversation |
| `return {"messages": [reply]}` | returns a **one-item list** | `add_messages` appends it. Returning the full list would also work but is wasteful |
| `InMemorySaver()` | the checkpointer | a dict. Swap for `PostgresSaver` in production — no other code changes |
| `compile(checkpointer=memory)` | attaches it | without this, every call starts empty |
| `{"configurable": {"thread_id": "alice"}}` | the config | in a real app this is the user id, chat id or ticket id |
| second `invoke` with only the new message | history is reloaded automatically | ⭐ this is the payoff: no manual message-list management |
| `get_state(alice).values` | the saved state | |
| `get_state(alice).next` | which node is pending | **how your backend knows whether a run is paused** (§7) |

**Real world:** a WhatsApp bot uses the phone number as `thread_id`; an internal tool uses the SSO user id; ticket automation uses the ticket id. Because state lives in Postgres, any container can serve the next message, and a deploy mid-conversation loses nothing.

## 5. Conditional edges — routing

A conditional edge is LangGraph's `if` statement. You give it a **routing function** that reads the state and returns the **name** of the next node.

```
                     ┌──► billing_agent ──┐
START → classify ────┼──► technical_agent ┼──► END
                     └──► fallback ───────┘
```

This "router" (or "supervisor") pattern is probably the most common production graph shape.

📌 Note the interaction with notebook 1: the classification uses **structured output** with a `Literal` field, so the route value *cannot* be `"Billing."` or `"i think billing"`. Reliable routing depends on reliable classification.

**Real-world routers:**

| Sphere | Routes to |
|---|---|
| Support | billing / technical / shipping / human escalation |
| Healthcare intake | triage nurse / GP appointment / emergency advice / pharmacy |
| Banking | dispute / card services / loan enquiry / fraud team |
| Internal assistant | HR docs / IT docs / finance docs / SQL analytics |
| Education | maths tutor / essay feedback / study planner |

In [ ]:
class RouteState(TypedDict):
    question: str
    topic: str
    answer: str

class Route(BaseModel):
    """Which specialist should handle this question?"""
    topic: Literal["billing", "technical", "other"] = Field(description="Best matching topic.")

router_llm = llm.with_structured_output(Route)     # Literal -> the value is a safe enum

def classify(state: RouteState) -> dict:
    topic = router_llm.invoke(f"Classify this customer question: {state['question']}").topic
    print("   routed to:", topic)
    return {"topic": topic}

def billing_agent(state: RouteState) -> dict:
    out = llm.invoke(f"You are a billing specialist. Be brief.\nQ: {state['question']}")
    return {"answer": out.content}

def technical_agent(state: RouteState) -> dict:
    out = llm.invoke(f"You are a support engineer. Give ONE concrete step.\nQ: {state['question']}")
    return {"answer": out.content}

def fallback_agent(state: RouteState) -> dict:
    return {"answer": "I'll pass this to a human colleague who can help."}

# THE ROUTING FUNCTION: reads state, returns the NAME of the next node (a string).
def pick_branch(state: RouteState) -> str:
    return {"billing": "billing", "technical": "technical"}.get(state["topic"], "fallback")
    #      ^ a dict lookup with a DEFAULT. A safe default branch is mandatory:
    #        never let an unexpected value crash the graph.

rb = StateGraph(RouteState)
for name, fn in [("classify", classify), ("billing", billing_agent),
                 ("technical", technical_agent), ("fallback", fallback_agent)]:
    rb.add_node(name, fn)

rb.add_edge(START, "classify")
rb.add_conditional_edges(
    "classify",                                # after THIS node...
    pick_branch,                               # ...call THIS function to decide...
    ["billing", "technical", "fallback"],      # ...and it must return one of THESE names
)                                              # (the list is used for validation + diagrams)
for n in ["billing", "technical", "fallback"]:
    rb.add_edge(n, END)                        # all branches converge on END

router_graph = rb.compile()

for q in ["I was charged twice last month",
          "The desktop app freezes when I export a PDF",
          "Do you sponsor open-source projects?"]:
    print(f"\nQ: {q}")
    print("A:", router_graph.invoke({"question": q, "topic": "", "answer": ""})["answer"][:150])

## 6. The ReAct agent as a graph

This is notebook 1's hand-written loop, rebuilt as a graph. **The logic is identical** — but now you get persistence, streaming, and state inspection for free.

```
START → agent ──(has tool_calls?)──► tools ──┐
          ▲                                   │
          └───────────────────────────────────┘   ⭐ THE CYCLE
          │
          └──(no tool_calls)──► END
```

Two prebuilt pieces do the boring work:

| Prebuilt | What it does | Replaces |
|---|---|---|
| `ToolNode(TOOLS)` | reads `.tool_calls` from the last `AIMessage`, runs the matching tools (in parallel if several), appends `ToolMessage`s with the correct `tool_call_id`s | ~15 lines of manual plumbing |
| `tools_condition` | a routing function: returns `"tools"` if the last message has tool calls, else `END` | the `if msg.tool_calls:` branch |

⭐ The edge `add_edge("tools", "agent")` is the one a LangChain chain **physically cannot express**. That single edge is why LangGraph exists.

In [ ]:
from langgraph.prebuilt import ToolNode, tools_condition

class AgentState(TypedDict):
    messages: Annotated[list, add_messages]

# NODE 1 - the "brain". One LLM call, which may request tools.
def agent_node(state: AgentState) -> dict:
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

# NODE 2 - the "hands". Prebuilt.
tool_node = ToolNode(TOOLS)

ab = StateGraph(AgentState)
ab.add_node("agent", agent_node)
ab.add_node("tools", tool_node)
ab.add_edge(START, "agent")
ab.add_conditional_edges("agent", tools_condition)   # -> "tools" or END
ab.add_edge("tools", "agent")                        # ⭐ THE CYCLE: results go back to the brain

agent_graph = ab.compile(checkpointer=InMemorySaver())

cfg = {"configurable": {"thread_id": "support-1"}}
out = agent_graph.invoke({"messages": [
    SystemMessage("You are Acme support. Always verify facts with tools before answering."),
    HumanMessage("Order B-99182 hasn't moved in 12 days. What are my options?"),
]}, cfg)

# Print the full trail, so you can SEE the loop: AI asks -> tool answers -> AI concludes.
for m in out["messages"]:
    label = {"system": "SYS ", "human": "USER", "ai": "AI  ", "tool": "TOOL"}.get(m.type, m.type)
    text = m.content or f"[wants tools: {[c['name'] for c in m.tool_calls]}]"
    print(f"{label} | {str(text)[:150]}")

# A follow-up on the SAME thread. The agent still has the whole context.
print("\n--- follow-up, same thread ---")
print(agent_graph.invoke(
    {"messages": [HumanMessage("Yes, reship it. Which policy allows that?")]}, cfg
)["messages"][-1].content)

In [ ]:
# The one-line version, for when the standard ReAct loop is all you need.
# In LangChain 1.x this lives in langchain.agents and runs on the LangGraph runtime.
from langchain.agents import create_agent

quick_agent = create_agent(
    model=llm,                      # a model object, or a string like "openai:gpt-4o-mini"
    tools=TOOLS,
    system_prompt="You are Acme support. Verify every fact with a tool. Be concise.",
    checkpointer=InMemorySaver(),   # memory, same as before
)

res = quick_agent.invoke(
    {"messages": [{"role": "user", "content": "Is liquid damage covered on keyboard C-55510?"}]},
    {"configurable": {"thread_id": "quick-1"}},
)
print(res["messages"][-1].content)

print("\nnodes in the prebuilt agent:", [n for n in quick_agent.get_graph().nodes if not n.startswith("__")])

### 📌 `create_agent` vs building it yourself

| Use `create_agent` | Build a `StateGraph` |
|---|---|
| the standard ReAct loop is enough (~90% of cases) | you need custom nodes or a custom topology |
| you want tools + memory + HITL fast | you need a gate in a **specific place**, or a critic cycle |
| prototyping | multiple cooperating agents, or business rules between steps |

Be able to explain **both**. Interviewers want to know you understand the loop, not just the shortcut. Use cases A and B below are exactly the situations where the prebuilt agent is not enough.

## 7. Human in the loop — the most important production feature

Any agent that can spend money, email a customer, or delete data needs an approval gate.

`interrupt()` **stops the graph** and hands control back to your application. Because the state is checkpointed, the pause can last a millisecond or three days, and can be resumed by a **completely different process** — a webhook from your admin UI, a Slack button, an email reply. The agent is not "waiting" in memory; it is *saved*.

```
run 1:  START → check_order → human_approval ⏸ ......... returns {"__interrupt__": [...]}
                                    │
                              [ checkpoint ]      ← state saved; process may exit
                                    │
run 2:  Command(resume={...}) → human_approval resumes → issue_refund → END
```

| Requirement | Detail |
|---|---|
| A checkpointer is **mandatory** | the pause literally *is* a saved checkpoint |
| The paused result contains `__interrupt__` | check for it rather than assuming; `.value` holds your payload |
| Resume with `Command(resume=…)` | and **no new input dict** |
| `get_state(cfg).next` | tells you whether a run is still waiting |

In [ ]:
from langgraph.types import interrupt, Command

class RefundState(TypedDict):
    order_id: str
    amount: float
    approved: Optional[bool]
    result: str

def check_order(state: RefundState) -> dict:
    """Look up how much money is at stake."""
    return {"amount": ORDERS.get(state["order_id"], {}).get("value", 0.0)}

def human_approval(state: RefundState) -> dict:
    """⭐ interrupt() STOPS the graph here and returns control to your application."""
    decision = interrupt({                      # whatever you pass is surfaced to the caller;
        "question": f"Approve a refund of ${state['amount']:.2f} for {state['order_id']}?",
        "order_id": state["order_id"],          #   render it in your UI as an approval card
        "amount": state["amount"],
    })
    # ⬇️ Execution RESUMES HERE on the next call. `decision` receives whatever you
    #    passed to Command(resume=...). The function is re-entered from the top,
    #    which is why interrupt() must be the first side-effect-free thing in the node.
    return {"approved": bool(decision.get("approve"))}

def issue_refund(state: RefundState) -> dict:
    """The irreversible action - concentrated in ONE node, AFTER the gate."""
    if state["approved"]:
        return {"result": f"Refund of ${state['amount']:.2f} issued for {state['order_id']}."}
    return {"result": f"Refund for {state['order_id']} was declined by a reviewer."}

rf = StateGraph(RefundState)
rf.add_node("check_order", check_order)
rf.add_node("human_approval", human_approval)
rf.add_node("issue_refund", issue_refund)
rf.add_edge(START, "check_order")
rf.add_edge("check_order", "human_approval")
rf.add_edge("human_approval", "issue_refund")
rf.add_edge("issue_refund", END)

refund_graph = rf.compile(checkpointer=InMemorySaver())   # ⚠️ REQUIRED for interrupt()

cfg = {"configurable": {"thread_id": "refund-B-99182"}}

# ---------- RUN 1: starts, then pauses ----------
paused = refund_graph.invoke(
    {"order_id": "B-99182", "amount": 0.0, "approved": None, "result": ""}, cfg
)
print("PAUSED. Your app would render this as an approval card:")
print("  ", paused["__interrupt__"][0].value)
print("   waiting at node:", refund_graph.get_state(cfg).next)
print("   nothing has been refunded. That is the point of the gate.")

# ---------- RUN 2: a human clicks Approve ----------
final = refund_graph.invoke(Command(resume={"approve": True}), cfg)   # no new input dict
print("\nRESUMED ->", final["result"])

# TRY IT: use a NEW thread_id and resume with {"approve": False} to see it decline.

### Where a gate is mandatory in the real world

| Sphere | Gated action |
|---|---|
| E-commerce | refunds, order cancellation, address changes |
| Banking / fintech | payments, limit increases, account closure, KYC overrides |
| Healthcare | anything clinical — AI drafts, a clinician signs |
| Legal | filing documents, sending client communications |
| HR | offers, terminations, salary changes |
| DevOps | production deploys, database migrations, scaling actions |
| Marketing | publishing to public channels |
| Insurance | claim approval above a threshold |

### 📌 Design the threshold, not the switch — and fail closed

Gating *everything* destroys the value of automation; gating *nothing* is reckless. So you set a risk threshold: auto-approve below it, escalate above it.

There is a trap in that, and it is worth remembering because it is a genuine production bug:

```python
amount = order_value(order_id)      # returns 0.0 when the order id is UNKNOWN

if amount <= 50:  auto_approve()    # ❌ every unparsed order id auto-approves!

known_small = 0 < amount <= 50      # ✅ unknown means ESCALATE, not "free"
if known_small:   auto_approve()
```

**When a safety check cannot be evaluated, take the safe branch.** That is what "fail closed" means, and use case A implements it.

## 8. Streaming and visualising

### Streaming modes

| `stream_mode` | Yields | Use for |
|---|---|---|
| `"updates"` | what each node returned | a progress UI: "Searching policies…", "Drafting reply…" |
| `"values"` | the full state after each node | debugging |
| `"messages"` | LLM tokens as generated, as `(token, metadata)` | the typing effect |
| `"debug"` | everything, very verbose | deep diagnosis |

**Real world:** perceived latency is a product feature. A 12-second agent feels broken in silence and fine if it narrates its steps. That is `stream_mode="updates"` piped to the UI.

In [ ]:
print("── stream_mode='updates' (progress per node) ──")
for chunk in agent_graph.stream(
    {"messages": [HumanMessage("What is the enterprise P1 response time?")]},
    {"configurable": {"thread_id": "stream-1"}},
    stream_mode="updates",
):
    for node, update in chunk.items():          # chunk is {node_name: update}
        last = update["messages"][-1]
        preview = last.content or f"[tools: {[c['name'] for c in last.tool_calls]}]"
        print(f"  node '{node}': {str(preview)[:100]}")

print("\n── stream_mode='messages' (token by token) ──")
for token, meta in agent_graph.stream(
    {"messages": [HumanMessage("In one sentence, what is the Pro tier response time?")]},
    {"configurable": {"thread_id": "stream-2"}},
    stream_mode="messages",
):
    if token.content:                            # tool-call chunks have empty content
        print(token.content, end="", flush=True)
print()

In [ ]:
# draw_mermaid() returns Mermaid text, which GitHub renders natively.
# Paste it into your README and your project has an architecture diagram that
# can never drift from the code, because it IS the code.
print(agent_graph.get_graph().draw_mermaid())

# Also available:
#   .draw_ascii()        needs: pip install grandalf
#   .draw_mermaid_png()  needs internet (renders via mermaid.ink)

## 9. Use case A — support triage agent with a critic loop and a refund gate

**The scenario:** a support team receives 800 tickets/day. An agent must read each one, classify it, look up the order, check policy, draft a reply, and escalate anything involving money. Median handle time is 11 minutes and the backlog never clears.

**Why this needs LangGraph and not a chain** — it uses two things a chain cannot express:

1. a **review → redraft cycle** (the draft is checked against a rubric and rewritten if it fails)
2. a **pause** for human approval before any refund

```
START → classify → enrich → draft → review ─┬─► finalize → END
                              ▲             │
                              └── revise ───┤   (max 2 attempts)
                                            │
                                            └─► human approval ⏸ → finalize
```

### The four patterns it demonstrates

| Pattern | Where | Why it matters |
|---|---|---|
| **Structured output for routing** | `classify` | deterministic branching, no prompt-parsing fragility |
| **Deterministic tool use** | `enrich` | when you already know which lookups are needed, plain Python calls are cheaper and cannot go wrong |
| **Generator–critic cycle** | `draft` ⇄ `review` | models spot flaws better than they avoid them; the critique is fed back so the loop converges |
| **Human gate, failing closed** | `approval` | nothing irreversible without a named approver |

In [ ]:
# ---------- THE STATE. Designing this IS designing the agent. ----------
# For each field ask: which node WRITES it, and which nodes READ it?
# If no node reads a field, delete it.
class TriageState(TypedDict):
    # input
    ticket_id: str
    ticket_text: str
    customer_tier: str

    # written by classify
    category: str
    urgency: int
    refund_requested: bool
    order_id: Optional[str]

    # written by enrich
    order_info: str
    policy_context: str

    # the cycle
    draft: str
    critique: str
    attempts: int            # ⚠️ a loop counter is MANDATORY. No counter, no exit.
    qa_passed: bool

    # the gate
    refund_approved: Optional[bool]

    # output
    final_reply: str

    # A CUSTOM REDUCER so every node can APPEND to the audit log without
    # overwriting what earlier nodes wrote.
    actions: Annotated[List[str], lambda old, new: old + new]

print("state fields:", len(TriageState.__annotations__))

In [ ]:
REFUND_LEDGER = []          # stands in for a payments API

@tool
def issue_refund_tool(order_id: str, amount: float, reason: str) -> str:
    """Issue a refund. Only ever called AFTER human approval."""
    REFUND_LEDGER.append({"order_id": order_id, "amount": amount, "reason": reason})
    return f"REFUND_ISSUED ${amount:.2f} for {order_id}"

def order_value(order_id: Optional[str]) -> float:
    """Helper, not a tool. Returns 0.0 for an UNKNOWN id - see the fail-closed note in §7."""
    return ORDERS.get((order_id or "").upper(), {}).get("value", 0.0)

# ---------- NODE 1: classify ----------
class Classification(BaseModel):
    """Structured triage of an inbound support ticket."""
    category: Literal["SHIPPING", "BILLING", "TECHNICAL", "RETURNS", "WARRANTY", "OTHER"] = Field(
        description="Primary category of the customer's problem.")
    urgency: int = Field(description="1 = routine, 5 = severe (outage, furious customer).", ge=1, le=5)
    order_id: Optional[str] = Field(default=None, description="Order id mentioned, else null.")
    refund_requested: bool = Field(description="True if the customer asks for money back.")

triage_classifier = llm.with_structured_output(Classification)

def classify_node(state: TriageState) -> dict:
    c = triage_classifier.invoke(
        f"Triage this ticket (customer tier: {state['customer_tier']}):\n{state['ticket_text']}"
    )
    print(f"  [1/6] classify -> {c.category}, urgency {c.urgency}, refund={c.refund_requested}")
    return {                                   # a partial update; unlisted keys stay as they are
        "category": c.category,
        "urgency": c.urgency,
        "order_id": c.order_id,
        "refund_requested": c.refund_requested,
        "attempts": 0,                         # initialise the loop counter HERE
        "actions": [f"classified as {c.category}"],
    }

# ---------- NODE 2: enrich ----------
def enrich_node(state: TriageState) -> dict:
    # 📌 DESIGN DECISION: we call the tools from plain Python instead of letting the LLM
    #    decide. We already KNOW which lookups are needed, so deterministic calls are
    #    cheaper, faster and cannot go wrong. Reserve model-chosen tool calling for
    #    genuinely open-ended steps.
    order_info = "No order id was provided."
    if state.get("order_id"):
        order_info = get_order_status.invoke({"order_id": state["order_id"]})

    policy = search_policies.invoke({"query": f"{state['category']}: {state['ticket_text'][:150]}"})
    print("  [2/6] enrich   -> order + policy fetched")
    return {"order_info": order_info, "policy_context": policy,
            "actions": ["fetched order and policy"]}

print("nodes 1-2 defined")

In [ ]:
# ---------- NODE 3: draft ----------
DRAFT_SYSTEM = """You are a senior customer support agent at Acme.

Write the reply the customer will receive:
- acknowledge the specific problem; apologise if they are upset
- state ONLY facts supported by the ORDER INFO and POLICY CONTEXT below
- give a concrete next step and quote the exact entitlement (days, %, $)
- promise nothing the policy does not allow
- under 120 words, no placeholders, sign off as "The Acme Support Team"

ORDER INFO
{order_info}

POLICY CONTEXT
{policy_context}

{critique_block}"""

def draft_node(state: TriageState) -> dict:
    # ⭐ THE KEY LINE OF THE WHOLE CYCLE.
    # On a retry we inject the reviewer's SPECIFIC objections into the drafting prompt.
    # That is what makes the loop CONVERGE. Without it the model rewrites at random
    # and can fail the same rubric item three times in a row.
    critique_block = ""
    if state.get("critique"):
        critique_block = (f"A reviewer REJECTED your previous draft. Fix these issues:\n"
                          f"{state['critique']}\n\nPrevious draft:\n{state['draft']}")

    msg = llm.invoke([
        SystemMessage(DRAFT_SYSTEM.format(order_info=state["order_info"],
                                          policy_context=state["policy_context"],
                                          critique_block=critique_block)),
        HumanMessage(f"Customer ticket:\n{state['ticket_text']}"),
    ])
    n = state.get("attempts", 0) + 1
    print(f"  [3/6] draft    -> attempt {n} ({len(msg.content.split())} words)")
    return {"draft": msg.content, "attempts": n, "actions": [f"drafted (attempt {n})"]}

# ---------- NODE 4: review (the critic) ----------
class Review(BaseModel):
    """QA verdict on a draft support reply."""
    approved: bool = Field(description="True only if the draft passes EVERY rubric item.")
    issues: str = Field(description="Numbered, specific, actionable fixes. 'none' if approved.")

reviewer = llm.with_structured_output(Review)

def review_node(state: TriageState) -> dict:
    # A second LLM call with a DIFFERENT role. Models are far better at spotting problems
    # in text they did not just write than at being perfect on the first pass.
    r = reviewer.invoke(
        "You are a strict QA reviewer for support replies. Reject if ANY item fails:\n"
        "1. Every policy claim is supported by the POLICY CONTEXT or ORDER INFO.\n"
        "2. It acknowledges the customer's actual problem.\n"
        "3. It gives a concrete next step, not a vague 'we'll look into it'.\n"
        "4. Under 130 words, no placeholders.\n"
        "5. It promises nothing beyond the stated policy.\n\n"
        f"POLICY CONTEXT\n{state['policy_context']}\n\n"
        f"ORDER INFO\n{state['order_info']}\n\n"
        f"DRAFT\n{state['draft']}"
    )
    print(f"  [4/6] review   -> {'approved' if r.approved else 'rejected: ' + r.issues[:60]}")
    return {"qa_passed": r.approved,
            "critique": "" if r.approved else r.issues,
            "actions": [f"QA {'pass' if r.approved else 'fail'}"]}

print("nodes 3-4 defined")

In [ ]:
# ---------- NODE 5: human approval (the money gate) ----------
AUTO_APPROVE_UNDER = 50.0        # a business threshold, not an AI decision

def approval_node(state: TriageState) -> dict:
    amount = order_value(state.get("order_id"))

    # FAIL CLOSED: auto-approve only when we KNOW the amount AND it is small.
    # order_value returns 0.0 for an unknown id - that means "unknown", not "free".
    known_small = 0 < amount <= AUTO_APPROVE_UNDER

    if known_small:
        print(f"  [5/6] approval -> auto-approved (${amount:.2f} <= ${AUTO_APPROVE_UNDER:.0f})")
        return {"refund_approved": True, "actions": [f"auto-approved ${amount:.2f}"]}

    print(f"  [5/6] approval -> PAUSED for a human (${amount:.2f})")
    decision = interrupt({
        "type": "refund_approval",
        "ticket_id": state["ticket_id"],
        "order_id": state.get("order_id"),
        "amount": amount,
        "customer_tier": state["customer_tier"],
        "draft_reply": state["draft"],
    })
    approved = bool(decision.get("approve"))
    return {"refund_approved": approved,
            "actions": [f"human {'approved' if approved else 'declined'} ${amount:.2f}"]}

# ---------- NODE 6: finalize ----------
def finalize_node(state: TriageState) -> dict:
    reply = state["draft"]
    actions = []

    if state.get("refund_requested"):
        if state.get("refund_approved"):
            amount = order_value(state.get("order_id"))
            # ⭐ The irreversible action happens in exactly ONE place, AFTER the gate.
            #    Concentrating side effects late and in one node keeps the blast radius auditable.
            actions.append(issue_refund_tool.invoke({
                "order_id": state.get("order_id") or "UNKNOWN",
                "amount": amount,
                "reason": f"{state['category']} issue, ticket {state['ticket_id']}",
            }))
            reply += (f"\n\nP.S. Your refund of ${amount:.2f} is approved and will appear "
                      f"in 5-7 business days.")
        elif state.get("refund_approved") is False:
            reply += ("\n\nRegarding the refund: a specialist is reviewing your case and "
                      "will contact you within one business day.")
            actions.append("refund declined, escalated to a specialist")

    # 📌 Escalation rules live in PYTHON, not in a prompt. "Enterprise tier or urgency >= 4
    #    escalates" is business policy: it must be deterministic, testable and auditable.
    #    The model handles language; code handles rules.
    if state.get("urgency", 0) >= 4 or state.get("customer_tier") == "enterprise":
        actions.append("flagged for human review")
    if not state.get("qa_passed"):
        actions.append("sent after max QA attempts, needs spot-check")

    print("  [6/6] finalize -> done")
    return {"final_reply": reply, "actions": actions}

print("nodes 5-6 defined")

In [ ]:
# ---------- WIRING: one routing function controls BOTH the cycle and the gate ----------
MAX_ATTEMPTS = 2

def after_review(state: TriageState) -> str:
    """Read this as three sentences."""
    # 1. Not good enough, and I still have attempts left -> try again.
    if not state.get("qa_passed") and state.get("attempts", 0) < MAX_ATTEMPTS:
        return "draft"
    # 2. Money is involved -> ask a human.
    if state.get("refund_requested"):
        return "approval"
    # 3. Otherwise -> send it.
    return "finalize"

tb = StateGraph(TriageState)
tb.add_node("classify", classify_node)
tb.add_node("enrich", enrich_node)
tb.add_node("draft", draft_node)
tb.add_node("review", review_node)
tb.add_node("approval", approval_node)
tb.add_node("finalize", finalize_node)

tb.add_edge(START, "classify")
tb.add_edge("classify", "enrich")
tb.add_edge("enrich", "draft")
tb.add_edge("draft", "review")
tb.add_conditional_edges("review", after_review, ["draft", "approval", "finalize"])
tb.add_edge("approval", "finalize")
tb.add_edge("finalize", END)

triage_graph = tb.compile(checkpointer=InMemorySaver())   # required for interrupt()

print(triage_graph.get_graph().draw_mermaid())

In [ ]:
TICKETS = {
    "T-1001": {"text": "Hi, I ordered a mechanical keyboard (C-55510) two days ago and just "
                       "want to know when it will arrive. No rush.", "tier": "free"},
    "T-1002": {"text": "This is unacceptable. Order B-99182 was supposed to arrive TEN DAYS "
                       "AGO and tracking hasn't moved once. I had to cancel a client "
                       "presentation. I want a full refund immediately.", "tier": "enterprise"},
    "T-1004": {"text": "I bought a laptop stand (A-10023) last week, it's unopened, and I've "
                       "changed my mind. Can I send it back? Please refund me.", "tier": "free"},
}

def run_ticket(ticket_id: str) -> dict:
    t = TICKETS[ticket_id]
    print(f"\n{'='*70}\nTICKET {ticket_id} ({t['tier']} tier)\n{'='*70}")
    return triage_graph.invoke(
        {"ticket_id": ticket_id, "ticket_text": t["text"], "customer_tier": t["tier"],
         "attempts": 0, "actions": []},
        {"configurable": {"thread_id": ticket_id}},   # thread_id = the ticket. Elegant:
    )                                                 # a crashed run resumes where it stopped.

# Ticket 1: no refund requested -> no human needed, straight through.
out1 = run_ticket("T-1001")
# .get() not [ ]: if the model unexpectedly flagged a refund, the run PAUSED and there
# is no final_reply yet. Never index a key that a branch might not have produced.
print("\nREPLY:\n", out1.get("final_reply", "(paused - awaiting approval)"))
print("\nAUDIT:", out1.get("actions", []))

In [ ]:
# Ticket 2: an angry enterprise refund above the threshold. The graph PAUSES.
out2 = run_ticket("T-1002")
cfg2 = {"configurable": {"thread_id": "T-1002"}}

if "__interrupt__" in out2:
    pending = out2["__interrupt__"][0].value
    print("\nPAUSED. Your app would render this as an approval card:")
    for k, v in pending.items():
        if k != "draft_reply":
            print(f"   {k}: {v}")
    print("\nDraft the customer would receive:\n")
    print(pending["draft_reply"])
    print("\nwaiting at node:", triage_graph.get_state(cfg2).next)
    print("Nothing was refunded, sent or charged.")
else:
    print("\nNo pause (no refund flagged, or under the threshold).")
    print(out2.get("final_reply", "")[:300])

In [ ]:
# A supervisor clicks Approve. In production this is a webhook arriving minutes or days
# later, possibly on a different server. The checkpointer is what makes that possible.

# get_state(...).next tells you whether the graph is still waiting. Empty tuple = finished.
if triage_graph.get_state(cfg2).next:
    final2 = triage_graph.invoke(
        Command(resume={"approve": True, "approver": "supervisor@acme.example"}), cfg2
    )
else:
    final2 = out2      # it already finished; nothing to resume

print("REPLY:\n", final2.get("final_reply", ""))
print("\nAUDIT:")
for a in final2.get("actions", []):
    print("  -", a)
print("\nLEDGER:", REFUND_LEDGER)

In [ ]:
# Ticket 3: a $45.50 refund - UNDER the $50 threshold, so it auto-approves with no pause.
out4 = run_ticket("T-1004")
print("\nREPLY:\n", out4.get("final_reply", "(paused)"))
print("\nAUDIT:", out4.get("actions", []))
print("\nLEDGER:", REFUND_LEDGER)

print("\n📌 The human/automation boundary is a business threshold you tune by RISK,")
print("   not an all-or-nothing choice.")

In [ ]:
# Measure it. Any agent you put on a resume should come with numbers.
def triage_metrics(states: dict) -> None:
    print(f"{'TICKET':<9}{'CATEGORY':<11}{'DRAFTS':<8}{'QA':<6}ESCALATED")
    done = []
    for tid, s in states.items():
        if "__interrupt__" in s:
            print(f"{tid:<9}{'PAUSED':<11}{'-':<8}{'-':<6}-")
            continue
        done.append(s)
        esc = "yes" if any("flagged" in a for a in s.get("actions", [])) else "no"
        print(f"{tid:<9}{s.get('category','?'):<11}{s.get('attempts',0):<8}"
              f"{'pass' if s.get('qa_passed') else 'FAIL':<6}{esc}")

    if done:
        auto = sum(1 for s in done if not any("flagged" in a for a in s.get("actions", [])))
        first = sum(1 for s in done if s.get("attempts") == 1)
        print(f"\nauto-resolved with no human flag : {auto}/{len(done)}")
        print(f"QA passed on the FIRST draft     : {first}/{len(done)}")
        print(f"refunds issued (all gated)       : {len(REFUND_LEDGER)}")

triage_metrics({"T-1001": out1, "T-1002": final2, "T-1004": out4})

print("\nThese are the metrics an operations manager asks for:")
print("  auto-resolution rate | first-pass QA rate | drafts per ticket | gated actions")

## 10. Use case B — research assistant with a plan → gather → check → write cycle

**The scenario:** an analyst asks a broad question. One search is not enough: the assistant must plan sub-questions, gather evidence for each, **check whether it has enough**, gather more if not, and only then write.

**Why this needs LangGraph:** the number of loops is not known in advance. The graph decides at runtime how many times to go round.

```
START → plan → gather ──► check ─┬─► write → END
                  ▲              │
                  └── not enough ┘   (max 3 rounds)
```

This is the **plan–execute** pattern, and it is what "deep research" features are built on. We research over the local policy documents so it runs offline-cheap, but the shape is identical with a web-search tool.

In [ ]:
class ResearchState(TypedDict):
    question: str
    sub_questions: List[str]
    evidence: Annotated[List[str], lambda old, new: old + new]   # append across rounds
    rounds: int                                                   # the loop counter
    enough: bool
    missing: str
    report: str

# ---------- NODE 1: plan ----------
class Plan(BaseModel):
    """A breakdown of a broad question into searchable sub-questions."""
    sub_questions: List[str] = Field(description="2-4 specific, independently searchable questions.")

planner = llm.with_structured_output(Plan)

def plan_node(state: ResearchState) -> dict:
    p = planner.invoke(
        "Break this question into 2-4 specific sub-questions that could each be answered "
        f"by searching a company policy document.\n\nQuestion: {state['question']}"
    )
    print("  [plan] sub-questions:")
    for q in p.sub_questions:
        print("     -", q)
    return {"sub_questions": p.sub_questions, "rounds": 0}

# ---------- NODE 2: gather ----------
def gather_node(state: ResearchState) -> dict:
    """Search for every sub-question. In a real agent this would be a web-search tool."""
    found = []
    for q in state["sub_questions"]:
        found.append(f"### {q}\n{search_policies.invoke({'query': q})}")
    n = state.get("rounds", 0) + 1
    print(f"  [gather] round {n}: {len(found)} searches")
    return {"evidence": found, "rounds": n}      # the reducer APPENDS to earlier rounds

print("nodes plan + gather defined")

In [ ]:
# ---------- NODE 3: check (the loop controller's input) ----------
class Sufficiency(BaseModel):
    """Is the gathered evidence enough to answer the question?"""
    enough: bool = Field(description="True if the evidence can fully answer the question.")
    missing: str = Field(description="What is still missing, or 'none'.")
    extra_questions: List[str] = Field(description="0-2 new sub-questions to close the gap.")

checker = llm.with_structured_output(Sufficiency)

def check_node(state: ResearchState) -> dict:
    joined = "\n\n".join(state["evidence"])[:6000]      # cap the prompt: evidence grows each round
    s = checker.invoke(
        f"QUESTION: {state['question']}\n\nEVIDENCE:\n{joined}\n\n"
        "Is this evidence sufficient to answer the question completely?"
    )
    print(f"  [check] {'enough' if s.enough else 'NOT enough: ' + s.missing[:60]}")
    return {
        "enough": s.enough,
        "missing": s.missing,
        # If more is needed, REPLACE the sub-questions with the gap-closing ones,
        # so the next gather round searches something NEW rather than repeating itself.
        "sub_questions": s.extra_questions if (not s.enough and s.extra_questions)
                         else state["sub_questions"],
    }

# ---------- NODE 4: write ----------
def write_node(state: ResearchState) -> dict:
    report = llm.invoke([
        SystemMessage(
            "Write a short briefing from the EVIDENCE only. Cite the [file.txt] labels. "
            "If something is not covered by the evidence, say so explicitly. Max 200 words."),
        HumanMessage(f"QUESTION: {state['question']}\n\nEVIDENCE:\n"
                     + "\n\n".join(state["evidence"])[:8000]),
    ]).content
    print("  [write] report ready")
    return {"report": report}

# ---------- THE LOOP CONTROLLER ----------
MAX_ROUNDS = 3

def after_check(state: ResearchState) -> str:
    if state["enough"]:
        return "write"
    if state.get("rounds", 0) >= MAX_ROUNDS:      # ⚠️ the bound. Without it: infinite loop.
        print("  [check] hit the round limit - writing with what we have")
        return "write"
    return "gather"                                # ⭐ loop back for another round

rs = StateGraph(ResearchState)
rs.add_node("plan", plan_node)
rs.add_node("gather", gather_node)
rs.add_node("check", check_node)
rs.add_node("write", write_node)
rs.add_edge(START, "plan")
rs.add_edge("plan", "gather")
rs.add_edge("gather", "check")
rs.add_conditional_edges("check", after_check, ["gather", "write"])
rs.add_edge("write", END)

research_graph = rs.compile()
print(research_graph.get_graph().draw_mermaid())

In [ ]:
question = ("A customer on the Enterprise plan received a damaged monitor 40 days after "
            "delivery and wants it replaced. What are we obliged to do, and how fast must "
            "we respond?")

print("QUESTION:", question, "\n")
research_out = research_graph.invoke({        # a distinct name: `out` is the agent result
    "question": question, "sub_questions": [], "evidence": [],
    "rounds": 0, "enough": False, "missing": "", "report": "",
})

print("\n" + "=" * 70)
print("BRIEFING\n")
print(research_out["report"])
print("\n" + "=" * 70)
print(f"rounds used: {research_out['rounds']} | "
      f"evidence pieces gathered: {len(research_out['evidence'])}")

### What to notice about use case B

1. **The graph decided how many rounds to run.** With a chain you would have to hard-code "always search twice".
2. **`MAX_ROUNDS` is not optional.** A self-judging loop with no bound is how people get a surprise bill. Every cycle needs a counter checked by the routing function.
3. **The gap-closing questions replace the old ones**, so round 2 searches something new. Re-running identical searches is the classic reason these loops fail to converge.
4. **The write node cites and admits gaps.** Same grounding discipline as RAG in notebook 1.

**Real-world versions of this pattern:** competitive research briefs, due-diligence summaries, incident post-mortems from logs and runbooks, literature reviews, and "deep research" features in consumer products.

## 11. Self-check and recap

In [ ]:
checks = []
def ck(name, condition):
    checks.append((name, bool(condition)))

ck("counter graph maths", graph.invoke({"value": 5, "log": ""})["value"] == 30)
ck("default reducer overwrites", ow.compile().invoke({"items": []})["items"] == ["b"])
ck("custom reducer appends", ap.compile().invoke({"items": []})["items"] == ["a", "b"])
ck("checkpointer kept the thread",
   len(chat_graph.get_state({"configurable": {"thread_id": "alice"}}).values["messages"]) >= 3)
ck("threads are isolated",
   len(chat_graph.get_state({"configurable": {"thread_id": "bob"}}).values["messages"]) <= 2)
ck("router produced an answer", bool(router_graph.invoke(
    {"question": "I was charged twice", "topic": "", "answer": ""})["answer"]))
ck("agent graph has a tools node", "tools" in agent_graph.get_graph().nodes)
ck("agent graph ran the tool loop", any(m.type == "tool" for m in out["messages"]))
ck("create_agent built a graph", "tools" in quick_agent.get_graph().nodes)
ck("refund graph resumed after interrupt", "issued" in final["result"])
ck("triage graph has 6 nodes",
   len([n for n in triage_graph.get_graph().nodes if not n.startswith("__")]) == 6)
ck("triage ticket 1 finished or paused",
   bool(out1.get("final_reply") or out1.get("__interrupt__")))
ck("triage ticket 2 resolved", bool(final2.get("final_reply") or final2.get("__interrupt__")))
ck("refund was gated and recorded", len(REFUND_LEDGER) >= 1)
ck("research loop stayed bounded", 1 <= research_out["rounds"] <= MAX_ROUNDS)
ck("research produced a report", len(research_out["report"]) > 50)

for name, ok in checks:
    print(f"{'ok  ' if ok else 'FAIL'} {name}")

failed = [n for n, ok in checks if not ok]
print("\n" + ("ALL CHECKS PASSED" if not failed else f"FAILED: {failed}"))

### Recap

| Concept | Code | The one-line version |
|---|---|---|
| State | `class S(TypedDict)` | the shared whiteboard |
| Node | `def n(state) -> dict` | a plain function returning a partial update |
| Edge | `add_edge("a", "b")` | fixed next step |
| Conditional edge | `add_conditional_edges("a", fn, [...])` | the `if` statement |
| Reducer | `Annotated[list, add_messages]` | append instead of overwrite |
| Checkpointer | `compile(checkpointer=...)` + `thread_id` | memory, durability, isolation |
| Tool loop | `ToolNode` + `tools_condition` + `add_edge("tools","agent")` | the ReAct cycle |
| Prebuilt agent | `create_agent(model, tools, ...)` | the fast path |
| Human gate | `interrupt()` + `Command(resume=…)` | pause for approval |
| Streaming | `stream_mode="updates"` | a responsive UI |
| Diagram | `get_graph().draw_mermaid()` | a README that can't drift |

### Troubleshooting

| Error | Cause | Fix |
|---|---|---|
| `GraphRecursionError` | a cycle with no exit | add a counter to state; check it in the routing function; or raise `recursion_limit` in the config |
| `interrupt()` seems ignored | no checkpointer | compile with `checkpointer=` and always pass a `thread_id` |
| Agent forgets everything between calls | a new `thread_id` each call | reuse the same `thread_id` for one conversation |
| `KeyError: '__interrupt__'` | you assumed the run paused | check `if "__interrupt__" in result:` |
| `KeyError: 'final_reply'` | a branch didn't produce that key | use `.get()` for any key a branch might skip |
| `KeyError: 'tool_call_id'` | a `ToolMessage` without a matching id | use `ToolNode`, or copy `call["id"]` exactly |
| A node's update is silently lost | you expected append, got overwrite | attach a reducer with `Annotated` |
| `InvalidUpdateError` | a node returned a key not in the state class | add the field to the `TypedDict` |

### The five sentences worth memorising

1. LangGraph = state + nodes + edges + conditional edges, plus a checkpointer and interrupts.
2. Reducers decide how updates merge; the default overwrites, `add_messages` appends.
3. `thread_id` + checkpointer = memory, durability and user isolation, for free.
4. Every cycle needs a bounded counter checked by the routing function.
5. Every irreversible action needs a human gate, and unknown must mean escalate.

### Next

**Notebook 3 — LangSmith.** You have now built things that loop, branch and pause. Next: how do you know any of it actually *works*, and how do you prove a change didn't break it?